In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys
# sys.path.append('/image-differential-annotator/')
sys.path.append('/projects/activities/kappsen-tmc/USERS/domans/differential-annotator-dev/image-differential-annotator')

from annotator.annotation import runAnnotation, jumpStart
from annotator.combineCDF import getDiscreteCombinedCDFofAllFeatures as PCMA
from annotator.stqutils import loadAd, preparePatchesWSI, getPatchRepresentation, inferProb, showProbImg

from tqdm import tqdm
import numpy as np
import pandas as pd
import pickle

import matplotlib.pyplot as plt

qs = np.linspace(0.05, 0.95, 10, endpoint=True)

classifierPaths = 'classifiers/'
if not os.path.exists(classifierPaths):
    os.makedirs(classifierPaths)

In [ ]:
outsPath = '/projects/activities/kappsen-tmc/USERS/domans/differential-annotator-dev/mIF/spatial-omics-tools/features/results/'
samples  = ['012-ae']

In [ ]:
# If L is None, then the annotator will do lazy loading of patches with Zarr, else load entire images at requested level L
L = None
ts = 112+1 # Center-to-center distance between tiles (not size of a tile!)
mpp = 0.5065161 # Image pixel size
N = 6 # patch size, e.g., 8 by 8 tiles
fname='downstream-example-data.h5ad'

# Load the STQ data for each sample
ads = {}
imgs = {}
for sample in tqdm(samples):
    ads[sample], imgs[sample] = loadAd(f'{outsPath}{sample}/', fname=fname, L=L, useInputImagePath=True)

# Prepare the patches coordinates for each sample and concatenate them into a single DataFrame
patchCoordinates = pd.concat([preparePatchesWSI(ads[sample].obs, N=N, spacing=ts/mpp, sample_id=sample) for sample in tqdm(samples)], axis=0) # N=8

# Get the patch SAMPLER representations for each sample and combine them into a single DataFrame
patchesCDFs = pd.concat([getPatchRepresentation(ads[sample], patchCoordinates.xs(sample, level='sample', axis=0), qs, sample_id=sample) for sample in tqdm(samples)], axis=0)

In [149]:
startParams = {}
jumpStart(patchCoordinates, patchesCDFs, imgs, startParams, ncols=4, nrows=1, ads=ads, L=1 if L is None else L,
        sh=int(ts/2)/mpp, figsize=(3, 3), seed=1, maxN=10**3, pyramidscale=2, equal_pca=False)

In [153]:
clf, plog, bp = {}, [], {}
try: bp.update({startParams['selected_patch']: 'positive'})
except: pass
runAnnotation(patchCoordinates, patchesCDFs, imgs, bp, clf, plog, ads=ads, qs=qs, minN=1,
            figsize=(5, 5), augFunc=PCMA, alpha=0.5, R=1, cmapColors=['lightcoral', 'blue'], # (5, 5)
            seed=0, randomness=0.5, L=1 if L is None else L, sh=int(ts/2)/mpp, startParams=startParams, pyramidscale=2)

In [ ]:
# To save the classifier
if False:
    with open(f'{classifierPaths}/clf-somename.pkl', 'wb') as tempfile:
        pickle.dump(clf['clf'], tempfile)

# To load the classifier back
if False:
    clf = {}
    with open(f'{classifierPaths}/clf-somename.pkl', 'rb') as tempfile:
        clf['clf'] = pickle.load(tempfile)

In [152]:
# To run inference on the entire slides
for i in range(len(samples)):
    infSample  = samples[i]
    x, y, p = inferProb(ads[infSample], clf['clf'], qs, tsize=ts/mpp, R=2)
    showProbImg(x, y, p, f=2, figsize=(3, 3), ts=ts, mpp=mpp, title=infSample)